# 02 · Reconstruct from the dense example data (offline)

Loads the bundled 30-position dense sweep (`data/example_1000nN`, a Multi75E-G probe
on PPLN at 1000 nN, IDS) and shows that the **model-light low-rank reconstruction**
recovers the full mode shape — resonance and antiresonance branches, sharp — from a
handful of D-optimally chosen positions, matching the full sweep's D-NS. This is the
offline counterpart of the on-instrument loop.

In [ ]:
import sys, os, glob, re
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from activemodemap.lowrank import (reconstruct_map, d_optimal_order,
                                   resonance_index, dns_from_map)

## Load the tune files into a position x frequency map

In [ ]:
data_dir = os.path.join('..', 'data', 'example_1000nN')
def parse_tune(path):
    txt = open(path, encoding='latin-1').read()
    m = txt.split('AmpBEGIN')[1] if 'AmpBEGIN' in txt else txt
    m = re.split(r'END', m)[0]
    v = np.array([float(x) for x in m.split() if re.match(r'^-?[\d.eE+-]+$', x)])
    n = (v.size // 3) * 3
    a = v[:n].reshape(-1, 3)
    return a[:, 0], a[:, 1], a[:, 2]   # freq, phase(deg), amp
files = sorted(glob.glob(os.path.join(data_dir, 'Tune_*.txt')),
               key=lambda s: int(re.search(r'_(\d+)\.txt', s).group(1)))
F, A, P = [], [], []
for fp in files:
    f, ph, a = parse_tune(fp); F.append(f); A.append(a); P.append(ph)
fg = F[0]
Z = np.array([np.interp(fg, f, a) for f, a in zip(F, A)]) * np.exp(
    1j * np.deg2rad(np.array([np.interp(fg, f, p) for f, p in zip(F, P)])))
npos = Z.shape[0]; x_grid = np.arange(npos, dtype=float)   # ~1 um steps
print(f'{npos} positions x {fg.size} frequencies, {fg[0]/1e3:.0f}-{fg[-1]/1e3:.0f} kHz')

## Reference D-NS from the full sweep, then reconstruct from few positions

In [ ]:
true = np.abs(Z)
ires_full = resonance_index(fg, Z)
dns_ref = dns_from_map(x_grid, Z, fg, ires_full)
print(f'reference D-NS (all {npos} positions): {dns_ref:.2f}')

RANK, K = 5, 6
sel = d_optimal_order(x_grid, RANK, n_select=K)
rec = reconstruct_map(x_grid, sel, Z[sel], RANK)
ires = resonance_index(fg, Z[sel])
dns = dns_from_map(x_grid, rec['Zrec'], fg, ires)
held = [i for i in range(npos) if i not in sel]
rmse = np.sqrt(np.mean((np.abs(rec['Zrec'][held]) - true[held])**2)) / true.max()
print(f'reconstruction from {K} positions {sorted(sel)}: '
      f'held-out RMSE {rmse*100:.1f}%, D-NS {dns:.2f}')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ext = [x_grid[0], x_grid[-1], fg[0]/1e3, fg[-1]/1e3]
vmin, vmax = np.log10(true+1e-6).min(), np.log10(true+1e-6).max()
ax[0].imshow(np.log10(true.T+1e-6), origin='lower', aspect='auto', extent=ext, cmap='viridis', vmin=vmin, vmax=vmax)
ax[0].axvline(dns_ref, color='#e34948', ls='--'); ax[0].set_title('measured (30 positions)')
ax[1].imshow(np.log10(np.abs(rec['Zrec']).T+1e-6), origin='lower', aspect='auto', extent=ext, cmap='viridis', vmin=vmin, vmax=vmax)
[ax[1].axvline(x_grid[s], color='w', lw=0.8) for s in sel]
ax[1].set_title(f'reconstructed from {K}')
ax[2].imshow((rec['std']/true.max()).T, origin='lower', aspect='auto', extent=ext, cmap='magma')
ax[2].set_title('uncertainty (1sigma)')
[a.set_xlabel('position (um)') for a in ax]; ax[0].set_ylabel('frequency (kHz)')
plt.tight_layout(); plt.show()